# 金融開放 API 實作 Notebook
## Agentic AI + Finance Data 教學示範

本 Notebook 示範 9 個常見的開放金融 API。

涵蓋內容：

1. `yfinance` — 全球股價與基本資訊
2. `Alpha Vantage` — 股價 / 外匯 / 加密貨幣 + 技術指標
3. `Finnhub` — 股價、新聞與財經資訊（含部分台股）
4. `FinMind` — 台股日線、籌碼與財報
5. `FRED` — 宏觀經濟資料（美國聯準會資料庫）
6. 世界銀行 `World Bank` API（透過 `wbdata`）— 跨國經濟指標
7. `CoinGecko` — 加密貨幣行情與市值
8. `NewsAPI` — 一般／金融新聞聚合
9. `Internet Search` - 使用DuckDuckGok進行網路搜索

> 注意：本 Notebook 側重教學結構與程式碼範例，實際執行時請自行填入各服務的 API Key。


## 環境準備

請先安裝本 Notebook 會用到的套件：


In [ ]:
# 建議在虛擬環境中執行
#!pip install yfinance finnhub-python FinMind fredapi wbdata pycoingecko requests pandas matplotlib ddgs

---
## 1. 使用 `yfinance` 取得全球股票資料

`yfinance` 是非常方便的 Yahoo Finance 非官方介面，無需 API Key，即可取得：

- 歷史股價（K 線資料）
- 股利與拆分
- 公司基本資訊
- 財報部分欄位

非常適合用來教學「如何快速取得金融資料」。

### 1.1 匯入套件

In [ ]:
import yfinance as yf
import pandas as pd

pd.set_option("display.max_columns", 10)

### 1.2 抓取個股歷史股價（以 AAPL 為例）

In [ ]:
# 建立一個股票物件
ticker = yf.Ticker("NVDA")

# 取得過去一年的日線股價
hist = ticker.history(period="1d", interval="1m")
hist.tail(100)

### 1.3 取得公司各項常用資訊 
參考 info_fields_zh_TW.csv

In [ ]:
info = ticker.info

print("公司名稱：", info.get("longName"))
print("產業別：", info.get("sector"))
print("市值：", f"{info.get('marketCap'):,}")

### 1.4 視覺化股價走勢

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(hist.index, hist["Close"])
plt.title("AAPL Close Price (Last 1Y)")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.grid(True)
plt.show()

---
## 2. 使用 Alpha Vantage 取得股價與技術指標

Alpha Vantage 提供：

- 股票 / 外匯 / 加密貨幣 時間序列
- 多種技術指標（SMA、EMA、RSI等）
- JSON 格式回應，適合示範 REST API 使用

> 需先到官網註冊並取得 `API_KEY`。  - https://www.alphavantage.co/
> 免費使用者只能使用部份功能

### 2.1 取得日線股價 + 20 日簡單移動平均線（SMA）

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

ALPHA_VANTAGE_KEY = "WIWVJQ86GE3FUYYC"  # TODO: 換成你自己的 key
symbol = "AAPL"

# 1) 取得日線股價
url_price = "https://www.alphavantage.co/query"
params_price = {
    "function": "TIME_SERIES_DAILY_ADJUSTED",
    "symbol": symbol,
    "outputsize": "compact",
    "apikey": ALPHA_VANTAGE_KEY,
}
r = requests.get(url_price, params=params_price)
j_price = r.json()

# 檢查有沒有拿到 Time Series (Daily)
if "Time Series (Daily)" not in j_price:
    print("⚠️ Price API 沒有回傳 'Time Series (Daily)'，實際回應如下：")
    print(j_price)
    raise SystemExit("請檢查 Alpha Vantage key、symbol 或是否被限流。")

data = j_price["Time Series (Daily)"]
df_price = pd.DataFrame.from_dict(data, orient="index", dtype=float)
df_price.index = pd.to_datetime(df_price.index)
df_price.sort_index(inplace=True)

print("✅ df_price.columns =", df_price.columns)


In [ ]:
# 2) 取得 20 日 SMA 指標
url_sma = "https://www.alphavantage.co/query"
params_sma = {
    "function": "SMA",
    "symbol": symbol,
    "interval": "daily",
    "time_period": 20,
    "series_type": "close",
    "apikey": ALPHA_VANTAGE_KEY,
}
r2 = requests.get(url_sma, params=params_sma)
j_sma = r2.json()

if "Technical Analysis: SMA" not in j_sma:
    print("⚠️ SMA API 沒有回傳 'Technical Analysis: SMA'，實際回應如下：")
    print(j_sma)
    raise SystemExit("請檢查 Alpha Vantage key 或是否被限流。")

data_sma = j_sma["Technical Analysis: SMA"]
df_sma = pd.DataFrame.from_dict(data_sma, orient="index", dtype=float)
df_sma.index = pd.to_datetime(df_sma.index)
df_sma.sort_index(inplace=True)

print("✅ df_sma.columns =", df_sma.columns)


### 2.2 合併股價與 SMA 並畫圖

In [ ]:
# 3) 找出價格表中代表 close 的欄位名
close_col_candidates = [c for c in df_price.columns if "close" in c.lower()]
if not close_col_candidates:
    raise SystemExit(f"在 df_price 欄位中找不到任何含 'close' 的欄位：{df_price.columns}")

close_col = close_col_candidates[0]
print("使用的收盤價欄位：", close_col)

# 4) 合併股價與 SMA
df_merge = df_price[[close_col]].join(df_sma, how="inner")
df_merge.rename(columns={close_col: "close", "SMA": "sma20"}, inplace=True)

print("✅ df_merge.columns =", df_merge.columns)
print(df_merge.tail())

# 5) 繪圖
plt.figure(figsize=(10, 4))
plt.plot(df_merge.index, df_merge["close"], label="Close")
plt.plot(df_merge.index, df_merge["sma20"], label="SMA 20")
plt.legend()
plt.title(f"{symbol} Close Price & SMA20 (Alpha Vantage)")
plt.grid(True)
plt.show()

---
## 3. 使用 Finnhub 取得股價與公司新聞

Finnhub 是金融資料 API 服務，支援：

- 即時或延遲股價
- 公司新聞、財報、財務指標
- 部分台股資料

> 需註冊並取得 `API_KEY`，再使用官方 `finnhub-python` 套件。 - https://finnhub.io/

### 3.1 初始化 Finnhub Client

In [ ]:
import finnhub
from datetime import datetime, timedelta

FINNHUB_KEY = "d4d9r41r01qovljop0e0d4d9r41r01qovljop0eg"  # TODO: 替換成你的 API Key

finnhub_client = finnhub.Client(api_key=FINNHUB_KEY)

### 3.2 取得即時報價

In [ ]:
quote = finnhub_client.quote("AAPL")
quote

### 3.3 取得最近 7 天的公司新聞

In [ ]:
today = datetime.utcnow().date()
week_ago = today - timedelta(days=7)

news = finnhub_client.company_news(
    "AAPL",
    _from=week_ago.strftime("%Y-%m-%d"),
    to=today.strftime("%Y-%m-%d"),
)

for n in news[:5]:
    print(n)
    print(n["datetime"], "-", n["headline"])
    print(n["url"])
    print("-" * 80)

---
## 4. 使用 FinMind 取得台股資料

[FinMind](https://finmind.github.io/) 是台灣開源金融資料專案：

- 台股日線價格
- 法人買賣超
- 籌碼結構
- 部分財報資料

適合用於台股量化、教學與研究。

⚠️ note 流量限制

限制 request 上限 : 600/hour。Limit amount of request, 300/hour.
至 FinMind 註冊並驗證信箱後，API 的 Request 加上 token 參數可以提高使用上限到 600/hr。
token 獲取方法，可在官網登入後獲取。

### 4.1 匯入並建立 DataLoader

In [ ]:
#!pip install tqdm

In [ ]:
from FinMind.data import DataLoader

dl = DataLoader()

### 4.2 取得台積電（2330）日線股價

In [ ]:
tw_stock = dl.taiwan_stock_daily(
    stock_id="2330",
    start_date="2024-01-01",
    end_date="2024-12-31",
)
tw_stock.head()

### 4.3 取得法人買賣超資料

In [ ]:
inst = dl.taiwan_stock_institutional_investors(
    stock_id="2330",
    start_date="2024-01-01",
    end_date="2024-12-31",
)
inst.tail()

---
## 5. 使用 FRED 取得宏觀經濟資料

FRED（Federal Reserve Economic Data）提供：

- 美國宏觀經濟指標
- 利率、通膨、失業率等
- 適合用於建立因子模型與宏觀情境分析

> 需註冊並取得 `API_KEY`，使用 `fredapi`。 - https://fredaccount.stlouisfed.org

### 5.1 初始化 FRED

In [ ]:
from fredapi import Fred

FRED_KEY = "5935e90899103768e900c694c1f0c25a"  # TODO: 替換成你的 API Key
fred = Fred(api_key=FRED_KEY)

### 5.2 取得 CPI 與 10 年期公債殖利率

In [ ]:
cpi = fred.get_series("CPIAUCSL")  # 消費者物價指數
dgs10 = fred.get_series("DGS10")    # 10 年期公債殖利率

print("CPI 最近 5 筆：")
print(cpi.tail())

print("\n10Y Treasury Yield 最近 5 筆：")
print(dgs10.tail())

### 5.3 合併並畫圖

In [ ]:
macro_df = pd.concat([cpi.rename("CPI"), dgs10.rename("10Y_Yield")], axis=1).dropna()

macro_df.tail()

In [ ]:
macro_df.plot(subplots=True, figsize=(10, 6), grid=True, title=["CPI", "10Y Treasury Yield"])
plt.tight_layout()
plt.show()

---
## 6. 使用 World Bank API（`wbdata`）取得全球經濟指標

世界銀行提供各國：

- GDP
- 人口
- 能源、教育、環境等指標

這裡示範使用 `wbdata` 抓取多國 GDP。

### 6.1 取得多國 GDP 資料

In [ ]:
import wbdata
import datetime
import pandas as pd
import matplotlib.pyplot as plt

# 1. 指標與國家設定
indicator = {"NY.GDP.MKTP.CD": "gdp"}  # GDP, current US$
countries = ["USA", "CHN", "JPN", "TWN"]

# 2. 設定時間範圍（可以用 datetime，也可以用字串）
start_date = datetime.datetime(2010, 1, 1)
end_date = datetime.datetime(2024, 12, 31)

# 3. 呼叫 wbdata.get_dataframe
df_gdp = wbdata.get_dataframe(
    indicators=indicator,
    country=countries,
    date=(start_date, end_date),  
    freq="Y",                     
    parse_dates=True,             
)

print("原始資料（前幾筆）：")
print(df_gdp.head())
print("\nIndex 型態：", type(df_gdp.index), "前幾個 index：", df_gdp.index[:5])
print("\nColumns：", df_gdp.columns)

In [ ]:
# df_gdp 是你剛剛印出那個 DataFrame
# 取出 gdp 這一欄，unstack 讓 country 變成欄位
gdp_wide = df_gdp["gdp"].unstack(level="country").sort_index()

print(gdp_wide.head())
print(gdp_wide.columns)

### 6.2 轉置並畫圖比較各國 GDP

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
gdp_wide.plot(ax=plt.gca(), grid=True)
plt.title("GDP by Country (Current US$)")
plt.ylabel("GDP (US$)")
plt.xlabel("Year")
plt.show()

---
## 7. 使用 CoinGecko 取得加密貨幣行情

CoinGecko 提供免費的加密貨幣資料：

- 各幣種即時價格
- 過去 N 天歷史價格
- 市值、成交量等

這裡使用 `pycoingecko`。

### 7.1 取得 BTC / ETH 即時價格

In [ ]:
from pycoingecko import CoinGeckoAPI

cg = CoinGeckoAPI()

prices = cg.get_price(ids=["bitcoin", "ethereum"], vs_currencies=["usd"])
prices

### 7.2 取得 BTC 過去 30 天價格

In [ ]:
btc_history = cg.get_coin_market_chart_by_id(
    id="bitcoin", vs_currency="usd", days=30
)

len(btc_history["prices"]), btc_history["prices"][:3]

In [ ]:
import numpy as np

btc_prices = btc_history["prices"]
dates = [pd.to_datetime(p[0], unit="ms") for p in btc_prices]
values = [p[1] for p in btc_prices]

plt.figure(figsize=(10, 4))
plt.plot(dates, values)
plt.title("Bitcoin Price (Last 30 Days)")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()

---
## 8. 使用 NewsAPI 聚合新聞（可結合 LLM 做摘要）

NewsAPI 提供：

- 一般新聞來源（BBC、Reuters、Bloomberg…）
- 可指定關鍵字、日期、語言
- 很適合拿來做「新聞摘要 Agent」示範

> 需註冊並取得 `API_KEY`。 - https://newsapi.ai
>
> 免費使用者：20000 request

### 8.1 搜尋 NVIDIA 相關新聞

In [ ]:
import requests

NEWSAPI_KEY = "7217389904814dd7b015826d1ea1e752"  # TODO: 替換成你的 API Key

url = "https://newsapi.org/v2/everything"
params = {
    "q": "NVIDIA",
    "language": "en",
    "sortBy": "publishedAt",
    "pageSize": 5,
    "apiKey": NEWSAPI_KEY,
}

resp = requests.get(url, params=params)
data = resp.json()

for article in data.get("articles", []):
    print(article["publishedAt"], "-", article["title"])
    print(article["url"])
    print("-" * 80)

---
## 9. Internet Search 
DuckDuckGo

In [ ]:
#!pip install ddgs

In [ ]:
from ddgs import DDGS

# 同步搜索文本结果
with DDGS() as ddgs:
    results = ddgs.text(query="精誠 資安", max_results=5)
    for r in results:
        print(r)

In [ ]:
from ddgs import DDGS

# 同步搜索文本结果
with DDGS() as ddgs:
    results = ddgs.news(query="精誠 資安", max_results=5)
    for r in results:
        print(r)